In [1]:
!pip install scikit-learn pandas numpy joblib imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\Hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, make_scorer
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import joblib
import json
import os
from collections import Counter

# -----------------------------------------
# CONFIG
# -----------------------------------------
DATA_PATH         = r"D:\FYP\Webapp\dataset\NF-UQ-NIDS-v2.csv"
OUTPUT_DIR        = r"D:\FYP\Webapp\output"
TARGET            = "Attack"
CHUNK_SIZE        = 500_000
MIN_CLASS_SAMPLES = 5
MIN_PER_CLASS     = 800
MAX_PER_CLASS     = 80_000
TOTAL_TARGET      = 1_200_000

# Target sample count per class after balancing.
# Minority classes are oversampled (SMOTE) to this number.
# Majority classes are undersampled (RandomUnderSampler) to this number.
# Result: all 21 classes have exactly EQUAL_TARGET samples in training.
EQUAL_TARGET      = 10_000

# Tuning controls
TUNE_SAMPLE_SIZE  = 250_000
N_ITER_SEARCH     = 12
CV_FOLDS          = 3

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------------------
# 1. DEFINE CANDIDATE FEATURES
# -----------------------------------------
CANDIDATE_FEATURES = [
    "IN_BYTES", "IN_PKTS", "OUT_BYTES", "OUT_PKTS",
    "FLOW_DURATION_MILLISECONDS", "DURATION_IN", "DURATION_OUT",
    "MIN_TTL", "MAX_TTL", "LONGEST_FLOW_PKT", "SHORTEST_FLOW_PKT",
    "MIN_IP_PKT_LEN", "MAX_IP_PKT_LEN",
    "SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES",
    "RETRANSMITTED_IN_BYTES", "RETRANSMITTED_IN_PKTS",
    "RETRANSMITTED_OUT_BYTES", "RETRANSMITTED_OUT_PKTS",
    "SRC_TO_DST_AVG_THROUGHPUT", "DST_TO_SRC_AVG_THROUGHPUT",
    "NUM_PKTS_UP_TO_128_BYTES", "NUM_PKTS_128_TO_256_BYTES",
    "NUM_PKTS_256_TO_512_BYTES", "NUM_PKTS_512_TO_1024_BYTES",
    "NUM_PKTS_1024_TO_1514_BYTES", "TCP_WIN_MAX_IN", "TCP_WIN_MAX_OUT",
    "ICMP_TYPE", "ICMP_IPV4_TYPE", "DNS_QUERY_ID", "DNS_QUERY_TYPE",
    "DNS_TTL_ANSWER", "FTP_COMMAND_RET_CODE",
    "L4_SRC_PORT", "L4_DST_PORT", "PROTOCOL", "TCP_FLAGS",
    "CLIENT_TCP_FLAGS", "SERVER_TCP_FLAGS",
]

# -----------------------------------------
# 2. PEEK HEADERS
# -----------------------------------------
print("Reading column names...")
all_cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
FEATURES, seen = [], set()
for f in CANDIDATE_FEATURES:
    if f in all_cols and f not in seen:
        FEATURES.append(f)
        seen.add(f)

USECOLS = FEATURES + [TARGET]
print(f"Total columns in CSV : {len(all_cols)}")
print(f"Feature columns used : {len(FEATURES)}  ->  loading {len(USECOLS)} of {len(all_cols)} columns")

# -----------------------------------------
# 3. PHASE 1 - fast class count
# -----------------------------------------
print("\nPhase 1: counting class distribution...")
total_counts = pd.Series(dtype=int)
for chunk in pd.read_csv(DATA_PATH, usecols=[TARGET], chunksize=CHUNK_SIZE):
    chunk[TARGET] = chunk[TARGET].replace({"Benign": "Normal"}).fillna("Unknown")
    total_counts  = total_counts.add(chunk[TARGET].value_counts(), fill_value=0)

total_counts = total_counts.astype(int).sort_values(ascending=False)
print(f"\nFull dataset rows : {total_counts.sum():,}")
print(total_counts.to_string())

# -----------------------------------------
# 4. ADAPTIVE PER-CLASS SAMPLE TARGETS
# -----------------------------------------
valid_classes = total_counts[total_counts >= MIN_CLASS_SAMPLES].index
valid_counts  = total_counts[valid_classes]

proportional     = (valid_counts / valid_counts.sum() * TOTAL_TARGET).round().astype(int)
per_class_target = (
    proportional
    .clip(lower=MIN_PER_CLASS, upper=MAX_PER_CLASS)
    .combine(valid_counts, min)
)
per_class_rate = (per_class_target / valid_counts).clip(upper=1.0)

print(f"\nAdaptive sample targets per class:")
summary = pd.DataFrame({
    "total_rows":     valid_counts,
    "target_samples": per_class_target,
    "rate_%":         (per_class_rate * 100).round(2),
}).sort_values("total_rows", ascending=False)
print(summary.to_string())
print(f"\nTotal target samples : {per_class_target.sum():,}")

# -----------------------------------------
# 5. PHASE 2 - chunked load with adaptive rates
# -----------------------------------------
print("\nPhase 2: sampling dataset in chunks...\n")
chunk_samples  = []
rows_processed = 0

for i, chunk in enumerate(pd.read_csv(
    DATA_PATH, usecols=USECOLS, chunksize=CHUNK_SIZE, low_memory=True
)):
    chunk[TARGET]   = chunk[TARGET].replace({"Benign": "Normal"}).fillna("Unknown")
    chunk[FEATURES] = chunk[FEATURES].fillna(0).replace([np.inf, -np.inf], 0)
    chunk = chunk[chunk[TARGET].isin(valid_classes)]
    if chunk.empty:
        continue

    sampled = chunk.groupby(TARGET, group_keys=False).apply(
        lambda x: x.sample(
            n=max(1, int(len(x) * per_class_rate.get(x.name, 0.01))),
            random_state=42
        )
    )
    chunk_samples.append(sampled)
    rows_processed += len(chunk)

    if (i + 1) % 10 == 0:
        so_far = sum(len(s) for s in chunk_samples)
        print(f"  -> {rows_processed:>12,} rows processed | sample so far: {so_far:,}")

df = pd.concat(chunk_samples, ignore_index=True)
print(f"\nSampled dataset: {df.shape}")
print("\nLabel distribution (before dedup):")
print(df[TARGET].value_counts().to_string())

# -----------------------------------------
# 5.5 DEDUPLICATE
# Removes identical rows so the same network
# flow cannot appear in both train and test.
# -----------------------------------------
rows_before = len(df)
df = df.drop_duplicates(subset=FEATURES)
rows_after  = len(df)
print(f"\nDeduplication: {rows_before:,} -> {rows_after:,} rows  ({rows_before - rows_after:,} duplicates removed)")
print("\nLabel distribution (after dedup):")
print(df[TARGET].value_counts().to_string())

# -----------------------------------------
# 6. ENCODE LABELS
# -----------------------------------------
le        = LabelEncoder()
y_encoded = le.fit_transform(df[TARGET])

label_mapping = {int(i): str(lbl) for i, lbl in enumerate(le.classes_)}
print("\nLabel mapping:")
print(json.dumps(label_mapping, indent=2))

# -----------------------------------------
# 7. PREPARE X AND y
# -----------------------------------------
X = df[FEATURES].values.astype(np.float32)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
y = y_encoded

# -----------------------------------------
# 8. TRAIN / TEST SPLIT  (before scaling and balancing)
# -----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"\nTraining samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")

# -----------------------------------------
# 9. SCALE  (fit on train only)
# -----------------------------------------
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# -----------------------------------------
# 9.5 EQUAL-CLASS BALANCING  (training data only)
#
# Step 1 - SMOTE: oversample every class below
#   EQUAL_TARGET up to EQUAL_TARGET samples.
# Step 2 - RandomUnderSampler: downsample every
#   class above EQUAL_TARGET down to EQUAL_TARGET.
#
# Result: all classes have exactly EQUAL_TARGET
# samples in training. Class weights are not
# needed because the data is already balanced.
# -----------------------------------------
train_counts_pre = Counter(y_train)
min_samples      = min(train_counts_pre.values())
k_neighbors      = min(5, min_samples - 1)  # SMOTE needs k+1 samples per class

over_strategy  = {cls: EQUAL_TARGET for cls, cnt in train_counts_pre.items() if cnt < EQUAL_TARGET}
under_strategy = {cls: EQUAL_TARGET for cls, cnt in train_counts_pre.items() if cnt > EQUAL_TARGET}

print(f"\nBalancing all classes to {EQUAL_TARGET:,} samples each...")
print(f"  Classes to oversample  (SMOTE)             : {len(over_strategy)}")
print(f"  Classes to undersample (RandomUnderSampler) : {len(under_strategy)}")
print(f"  k_neighbors for SMOTE  : {k_neighbors}")

steps = []
if over_strategy:
    steps.append(("smote", SMOTE(sampling_strategy=over_strategy,
                                  random_state=42, k_neighbors=k_neighbors)))
if under_strategy:
    steps.append(("under", RandomUnderSampler(sampling_strategy=under_strategy,
                                               random_state=42)))

if steps:
    pipeline = Pipeline(steps)
    X_train, y_train = pipeline.fit_resample(X_train, y_train)

print(f"\nTraining set after balancing : {len(X_train):,} samples")
print("\nClass distribution after balancing:")
after_counts = Counter(y_train)
for cls in sorted(after_counts):
    before = train_counts_pre.get(cls, 0)
    after  = after_counts[cls]
    if after > before:
        tag = f"  +{after - before:,} synthetic"
    elif after < before:
        tag = f"  -{before - after:,} removed"
    else:
        tag = ""
    print(f"  {label_mapping[cls]:20s}: {before:>7,} -> {after:>7,}{tag}")

# -----------------------------------------
# 10. HYPERPARAMETER TUNING (macro-F1)
# Classes are now balanced so no class weights needed.
# -----------------------------------------
print("\nTuning Random Forest for macro-F1...")

if len(X_train) > TUNE_SAMPLE_SIZE:
    _, X_tune, _, y_tune = train_test_split(
        X_train, y_train,
        test_size=TUNE_SAMPLE_SIZE,
        random_state=42,
        stratify=y_train
    )
else:
    X_tune, y_tune = X_train, y_train

print(f"Tuning set size: {len(X_tune):,}")

base_rf = RandomForestClassifier(
    class_weight=None,
    random_state=42,
    n_jobs=-1,
    oob_score=False
)

param_dist = {
    "n_estimators":      [200, 300],
    "max_depth":         [14, 16, 18],
    "min_samples_split": [20, 40],
    "min_samples_leaf":  [10, 20, 40],
    "max_features":      ["sqrt"],
    "max_samples":       [0.6, 0.75],
}

cv       = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
macro_f1 = make_scorer(f1_score, average="macro")

search = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_dist,
    n_iter=N_ITER_SEARCH,
    scoring=macro_f1,
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=False
)
search.fit(X_tune, y_tune)

best_params = search.best_params_
print("Best CV macro-F1:", round(search.best_score_, 4))
print("Best params:", best_params)

# -----------------------------------------
# 11. TRAIN FINAL MODEL
# -----------------------------------------
print("\nTraining final Random Forest...")
rf_model = RandomForestClassifier(
    **best_params,
    class_weight=None,
    random_state=42,
    n_jobs=-1,
    oob_score=True,
    verbose=1
)
rf_model.fit(X_train, y_train)

# -----------------------------------------
# 12. OVERFITTING DIAGNOSTIC
# -----------------------------------------
train_acc = rf_model.score(X_train, y_train)
test_acc  = rf_model.score(X_test,  y_test)
oob_acc   = rf_model.oob_score_

print(f"\n-- Overfitting Check --")
print(f"  Train accuracy : {train_acc:.4f}")
print(f"  OOB   accuracy : {oob_acc:.4f}   (should be close to test)")
print(f"  Test  accuracy : {test_acc:.4f}")
gap = train_acc - test_acc
print(f"  Train-Test gap : {gap:.4f}  {'WARNING: possible overfit' if gap > 0.05 else 'OK'}")

# -----------------------------------------
# 13. EVALUATE
# -----------------------------------------
y_pred = rf_model.predict(X_test)

print("\n-- Classification Report --")
print(classification_report(y_test, y_pred, target_names=list(label_mapping.values())))

print("\n-- Confusion Matrix --")
cm_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=list(label_mapping.values()),
    columns=list(label_mapping.values())
)
print(cm_df)

# -----------------------------------------
# 14. FEATURE IMPORTANCE
# -----------------------------------------
importances = pd.DataFrame({
    "feature":    FEATURES,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n-- Top 10 Most Important Features --")
print(importances.head(10).to_string(index=False))

# -----------------------------------------
# 15. SAVE ALL OUTPUT FILES
# -----------------------------------------
joblib.dump(rf_model,  os.path.join(OUTPUT_DIR, "rf_anomaly_model.pkl")); print("\nModel saved.")
joblib.dump(scaler,    os.path.join(OUTPUT_DIR, "scaler.pkl"));           print("Scaler saved.")
joblib.dump(FEATURES,  os.path.join(OUTPUT_DIR, "feature_list.pkl"));     print("Feature list saved.")

with open(os.path.join(OUTPUT_DIR, "label_mapping.json"), "w") as f:
    json.dump(label_mapping, f, indent=2)
print("Label mapping saved.")

importances.to_csv(os.path.join(OUTPUT_DIR, "feature_importance.csv"), index=False)
print("Feature importance saved.")

print(f"\n-- All files saved to {OUTPUT_DIR} --")
print(os.listdir(OUTPUT_DIR))

# -----------------------------------------
# 16. INFERENCE FUNCTION
# -----------------------------------------
def predict_from_wazuh_log(parsed_log: dict) -> dict:
    features  = joblib.load(os.path.join(OUTPUT_DIR, "feature_list.pkl"))
    _scaler   = joblib.load(os.path.join(OUTPUT_DIR, "scaler.pkl"))
    model     = joblib.load(os.path.join(OUTPUT_DIR, "rf_anomaly_model.pkl"))
    with open(os.path.join(OUTPUT_DIR, "label_mapping.json")) as f:
        label_map = json.load(f)

    fv = np.array([parsed_log.get(feat, 0) for feat in features], dtype=np.float32).reshape(1, -1)
    fv = np.nan_to_num(fv, nan=0.0, posinf=0.0, neginf=0.0)
    fv = _scaler.transform(fv)
    prediction    = model.predict(fv)[0]
    probabilities = model.predict_proba(fv)[0]

    normal_index = next((k for k, v in label_map.items() if v == "Normal"), None)
    normal_prob  = float(probabilities[int(normal_index)]) if normal_index else None

    return {
        "prediction":         label_map[str(prediction)],
        "confidence":         round(float(max(probabilities)), 4),
        "normal_probability": round(normal_prob, 4) if normal_prob else None,
        "probabilities": {label_map[str(i)]: round(float(p), 4) for i, p in enumerate(probabilities)},
    }

# -- Test: SSH brute-force simulation --
sample_log = {
    "IN_BYTES": 12000, "OUT_BYTES": 400, "IN_PKTS": 200, "OUT_PKTS": 15,
    "FLOW_DURATION_MILLISECONDS": 150, "PROTOCOL": 6,
    "TCP_FLAGS": 2, "L4_SRC_PORT": 54321, "L4_DST_PORT": 22,
    "MIN_TTL": 64, "MAX_TTL": 64,
}
print("\n-- Sample Inference --")
print(json.dumps(predict_from_wazuh_log(sample_log), indent=2))

Reading column names...
Total columns in CSV : 46
Feature columns used : 40  ->  loading 41 of 46 columns

Phase 1: counting class distribution...

Full dataset rows : 44,141,731
Attack
Normal            14620157
DDoS              12631341
DoS               10383877
scanning           2196755
Reconnaissance     1530158
xss                1426045
password            670208
injection           398233
Bot                  83371
Brute Force          72105
Infilteration        67450
Exploits             18376
Fuzzers              12952
Backdoor             11095
Generic               9536
mitm                  4383
ransomware            2002
Theft                 1404
Analysis              1318
Shellcode              854
Worms                  110
Unknown                  1

Adaptive sample targets per class:
                total_rows  target_samples  rate_%
Attack                                            
Normal            14620157           80000    0.55
DDoS              12631341     

C:\Users\Hp\AppData\Local\Temp\ipykernel_21520\1984463696.py:128: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = chunk.groupby(TARGET, group_keys=False).apply(
C:\Users\Hp\AppData\Local\Temp\ipykernel_21520\1984463696.py:128: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = chunk.groupby(TARGET, group_keys=False).apply(
C:\Users\Hp\AppData\Local\Temp\ipykernel_21520\1984463696.py:128: FutureWarning: D

  ->    5,000,000 rows processed | sample so far: 47,759


C:\Users\Hp\AppData\Local\Temp\ipykernel_21520\1984463696.py:128: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = chunk.groupby(TARGET, group_keys=False).apply(
C:\Users\Hp\AppData\Local\Temp\ipykernel_21520\1984463696.py:128: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = chunk.groupby(TARGET, group_keys=False).apply(
C:\Users\Hp\AppData\Local\Temp\ipykernel_21520\1984463696.py:128: FutureWarning: D

  ->   10,000,000 rows processed | sample so far: 95,514


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:   10.5s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   11.7s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.3s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    1.8s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    2.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.7s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.8s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.



-- Overfitting Check --
  Train accuracy : 0.9463
  OOB   accuracy : 0.9412   (should be close to test)
  Test  accuracy : 0.8985
  Train-Test gap : 0.0478  OK


[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.7s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.8s finished



-- Classification Report --
                precision    recall  f1-score   support

      Analysis       0.70      0.99      0.82        67
      Backdoor       0.97      0.99      0.98       142
           Bot       1.00      1.00      1.00       423
   Brute Force       0.97      0.98      0.97       383
          DDoS       0.99      0.98      0.98     15973
           DoS       0.97      0.95      0.96     15969
      Exploits       0.79      0.80      0.79       143
       Fuzzers       0.72      0.93      0.81       141
       Generic       0.94      0.80      0.86       132
 Infilteration       0.06      0.58      0.11       359
        Normal       0.97      0.71      0.82     15939
Reconnaissance       0.85      0.92      0.88      8290
     Shellcode       0.94      0.98      0.96       153
         Theft       0.76      0.99      0.86       149
         Worms       0.79      1.00      0.88        22
     injection       0.73      0.85      0.78      2155
          mitm    

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
